# BTCUSDT Orderflow Backtest: OrderBookImbalance Strategy

This notebook demonstrates an orderflow-based trading strategy using real BTCUSDT perpetual futures order book data from Binance.

## What is Orderflow Trading?

Orderflow trading analyzes the real-time flow of buy and sell orders in the market, rather than relying solely on derived indicators like moving averages or RSI.

Key concepts:
- **Order Book**: A live list of all outstanding buy (bid) and sell (ask) orders at various price levels
- **Bid/Ask Size Imbalance**: When one side of the book has significantly more volume than the other, it suggests directional pressure
- **Quote Tick**: A snapshot of the best bid and best ask prices with their respective sizes

Unlike traditional OHLC-based analysis which aggregates price action into candles, orderflow works with the raw market data ticks, providing a more granular view of supply and demand dynamics.

## OrderBookImbalance Strategy

Built-in example in NautilusTrader. The logic:

```
ratio = min(bidSize, askSize) / max(bidSize, askSize)

if ratio < trigger_imbalance_ratio (default 0.20):
    # Market is heavily skewed to one side
    if bidSize > askSize:
        # More buyers → price may rise → BUY at ask
        submit FOK BUY limit order at best ask
    else:
        # More sellers → price may fall → SELL at bid
        submit FOK SELL limit order at best bid
```

**FOK (Fill-Or-Kill)**: The order must be filled entirely or is cancelled immediately. This prevents partial fills.

**Cooldown**: Prevents repeated triggers within a short time window.

**Data requirement**: This strategy needs **quote ticks** (best bid/ask with sizes) from the order book, not just trade ticks. The real bid/ask data is critical for meaningful orderflow backtesting.

In [ ]:
import io
import zipfile
from decimal import Decimal

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests

from nautilus_trader.adapters.binance import BINANCE_VENUE
from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.examples.strategies.orderbook_imbalance import (
    OrderBookImbalance,
    OrderBookImbalanceConfig,
)
from nautilus_trader.model.currencies import USDT
from nautilus_trader.model.data import QuoteTick
from nautilus_trader.model.enums import AccountType, BookType, OmsType
from nautilus_trader.model.identifiers import TraderId
from nautilus_trader.model.objects import Money, Price, Quantity
from nautilus_trader.test_kit.providers import TestInstrumentProvider

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 5)
print('Libraries loaded.')

In [ ]:
# === Download BTCUSDT bookTicker data from Binance Public Data ===
# We use the daily file for 2024-03-15 and sample the first ~30 minutes.

INSTRUMENT = TestInstrumentProvider.btcusdt_perp_binance()
print(f'Instrument: {INSTRUMENT.id}')
print(f'Price precision: {INSTRUMENT.price_precision}')
print(f'Size precision: {INSTRUMENT.size_precision}')

DATE = '2024-03-15'
URL = (
    'https://data.binance.vision/data/futures/um/daily/bookTicker/BTCUSDT/'
    f'BTCUSDT-bookTicker-{DATE}.zip'
)

print(f'\nDownloading bookTicker data from Binance Public Data...')
resp = requests.get(URL)
z = zipfile.ZipFile(io.BytesIO(resp.content))
csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]

MAX_ROWS = 100_000  # ~30 min of bookTicker data
df = pd.read_csv(
    z.open(csv_name),
    nrows=MAX_ROWS,
    dtype={
        'best_bid_price': 'float64',
        'best_bid_qty': 'float64',
        'best_ask_price': 'float64',
        'best_ask_qty': 'float64',
        'transaction_time': 'int64',
    },
)
print(f'Loaded {len(df)} rows of bookTicker data')
print(f'Time range: {pd.Timestamp(df["transaction_time"].min(), unit="ms")}')
print(f'             to {pd.Timestamp(df["transaction_time"].max(), unit="ms")}')

# Convert to NautilusTrader QuoteTick list
quote_ticks = []
for _, row in df.iterrows():
    ts = int(row['transaction_time']) * 1_000_000  # ms -> ns
    quote_ticks.append(QuoteTick(
        instrument_id=INSTRUMENT.id,
        bid_price=Price(row['best_bid_price'], precision=INSTRUMENT.price_precision),
        ask_price=Price(row['best_ask_price'], precision=INSTRUMENT.price_precision),
        bid_size=Quantity(row['best_bid_qty'], precision=INSTRUMENT.size_precision),
        ask_size=Quantity(row['best_ask_qty'], precision=INSTRUMENT.size_precision),
        ts_event=ts,
        ts_init=ts,
    ))

print(f'Converted to {len(quote_ticks)} QuoteTick objects')
del df  # free memory

In [ ]:
# === Run OrderBookImbalance backtest ===

config = BacktestEngineConfig(
    trader_id=TraderId('BACKTESTER-001'),
    logging=LoggingConfig(log_level='WARN', use_pyo3=False),
)

engine = BacktestEngine(config=config)
engine.add_venue(
    venue=BINANCE_VENUE,
    oms_type=OmsType.NETTING,
    book_type=BookType.L1_MBP,
    account_type=AccountType.MARGIN,
    base_currency=None,
    starting_balances=[Money(100_000.0, USDT)],
)

engine.add_instrument(INSTRUMENT)
engine.add_data(quote_ticks)
print(f'Added {len(quote_ticks)} quote ticks')

strategy_config = OrderBookImbalanceConfig(
    instrument_id=INSTRUMENT.id,
    max_trade_size=Decimal('0.01'),       # 0.01 BTC per trade
    trigger_min_size=0.1,                  # min larger side to trigger
    trigger_imbalance_ratio=0.20,          # ratio < 0.20 triggers
    min_seconds_between_triggers=5.0,      # cooldown
    use_quote_ticks=True,                   # use L1_MBP quote ticks
    dry_run=False,
)

strategy = OrderBookImbalance(config=strategy_config)
engine.add_strategy(strategy=strategy)

print('\nRunning backtest...')
engine.run()
print('Backtest complete.')

## Results Interpretation Guide

The charts below show:
1. **Equity Curve**: How the account balance evolved over the trading session
2. **PnL per Trade**: Individual trade performance
3. **Bid/Ask Sizes**: The raw order book liquidity at the top level over time
4. **Imbalance Ratio**: `min(bid, ask) / max(bid, ask)` — when this drops below 0.20, a trigger fires

Note: This is a simple example strategy on a short (~30 min) window. The purpose is to demonstrate the orderflow backtesting pipeline, not to generate consistent profits.

In [ ]:
# === Account Equity Curve ===
account_report = engine.trader.generate_account_report(BINANCE_VENUE)
usdt_bal = account_report[account_report['currency'] == 'USDT'].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(usdt_bal.index, usdt_bal['total'], color='#2196F3', linewidth=1.5, label='USDT Balance')
ax.fill_between(usdt_bal.index, usdt_bal['total'], alpha=0.15, color='#2196F3')
ax.set_title('Account Equity Curve (USDT)', fontsize=14, fontweight='bold')
ax.set_ylabel('Balance (USDT)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.tight_layout()
plt.show()

usdt_start = float(usdt_bal.iloc[0]['total'])
usdt_end = float(usdt_bal.iloc[-1]['total'])
print(f'Start: {usdt_start:,.2f} USDT  ->  End: {usdt_end:,.2f} USDT')
print(f'Net PnL: {usdt_end - usdt_start:+.2f} USDT')

In [ ]:
# === PnL per Trade ===
positions_report = engine.trader.generate_positions_report().reset_index()

def money_to_float(v):
    if hasattr(v, 'as_double'):
        return float(v.as_double())
    return float(str(v).split()[0])

positions_report['pnl'] = positions_report['realized_pnl'].apply(money_to_float)

if len(positions_report) > 0:
    colors = ['#4CAF50' if p > 0 else '#F44336' for p in positions_report['pnl']]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.bar(range(len(positions_report)), positions_report['pnl'], color=colors, edgecolor='white')
    ax1.axhline(0, color='black', linewidth=0.5)
    ax1.set_title('PnL per Trade', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Trade #')
    ax1.set_ylabel('PnL (USDT)')
    ax1.grid(True, alpha=0.3, axis='y')

    cumsum = positions_report['pnl'].cumsum()
    ax2.plot(range(len(cumsum)), cumsum, color='#FF9800', linewidth=2, marker='o', markersize=4)
    ax2.fill_between(range(len(cumsum)), cumsum, alpha=0.15, color='#FF9800')
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.set_title('Cumulative PnL', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Trade #')
    ax2.set_ylabel('Cumulative PnL (USDT)')
    ax2.grid(True, alpha=0.3)

    wins = len(positions_report[positions_report['pnl'] > 0])
    total = len(positions_report)
    print(f'Trades: {total} | Wins: {wins} ({wins/total*100:.1f}%)')
    print(f'Total PnL: {positions_report["pnl"].sum():+.2f} USDT')
    print(f'Avg PnL: {positions_report["pnl"].mean():+.4f} USDT')
    plt.tight_layout()
    plt.show()
else:
    print('No trades were executed (no triggers fired).')

In [ ]:
# === Orderflow Visualization: Bid/Ask & Imbalance ===
# Re-read a fresh sample of the raw data for plotting
z = zipfile.ZipFile(io.BytesIO(resp.content))
csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
raw = pd.read_csv(z.open(csv_name), nrows=MAX_ROWS)
raw['ts'] = pd.to_datetime(raw['transaction_time'], unit='ms')

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Bid/Ask sizes
axes[0].plot(raw['ts'], raw['best_bid_qty'], alpha=0.5, linewidth=0.5, label='Bid Size', color='#4CAF50')
axes[0].plot(raw['ts'], raw['best_ask_qty'], alpha=0.5, linewidth=0.5, label='Ask Size', color='#F44336')
axes[0].set_ylabel('Size (BTC)')
axes[0].set_title('Top-of-Book Bid/Ask Sizes', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: Imbalance ratio
smaller = raw[['best_bid_qty', 'best_ask_qty']].min(axis=1)
larger = raw[['best_bid_qty', 'best_ask_qty']].max(axis=1)
raw['ratio'] = smaller / larger
axes[1].plot(raw['ts'], raw['ratio'], color='#9C27B0', linewidth=0.5)
axes[1].axhline(0.20, color='black', linewidth=1, linestyle='--', label='Trigger threshold (0.20)')
axes[1].set_ylabel('Ratio')
axes[1].set_title('Imbalance Ratio (smaller / larger)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Panel 3: Bid-Ask spread (price)
axes[2].plot(raw['ts'], raw['best_bid_price'], alpha=0.7, linewidth=0.5, label='Bid', color='#4CAF50')
axes[2].plot(raw['ts'], raw['best_ask_price'], alpha=0.7, linewidth=0.5, label='Ask', color='#F44336')
axes[2].set_ylabel('Price (USDT)')
axes[2].set_title('Bid/Ask Prices', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Overlay trade entry/exit markers if available
if len(positions_report) > 0:
    for _, pos in positions_report.iterrows():
        entry_time = pd.Timestamp(pos['ts_opened'])
        exit_time = pd.Timestamp(pos['ts_closed'])
        entry_px = float(pos['avg_px_open'])
        exit_px = float(pos['avg_px_close'])
        pnl = pos['pnl']
        color = '#4CAF50' if pnl > 0 else '#F44336'
        marker = '^' if str(pos['entry']).strip() == 'BUY' else 'v'
        axes[2].scatter(entry_time, entry_px, marker=marker, color=color, s=80,
                       zorder=5, edgecolors='black', linewidth=0.5)
        axes[2].scatter(exit_time, exit_px, marker='o', color=color, s=60,
                       zorder=5, edgecolors='black', linewidth=0.5, alpha=0.7)

axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.tight_layout()
plt.show()

In [ ]:
engine.dispose()
print('Done.')

## Next Steps

**Using more data**:
- Increase `MAX_ROWS` or download multiple daily files
- Merge consecutive days for multi-session backtesting

**Live data**:
- Use NautilusTrader's Binance adapter for real-time order book streaming
- Replace `use_quote_ticks=True` with direct `BookType.L2_MBP` subscription for full order book depth

**Strategy customization**:
- Adjust `trigger_imbalance_ratio` to see how sensitivity affects performance
- Add position sizing based on account equity
- Incorporate stop-loss logic
- Add volume-weighted or time-weighted trigger confirmation

**Data source alternative**:
- Binance Public Data also provides **depth snapshots** (L2 order book data) for more advanced orderflow analysis
- Path: `data/futures/um/daily/depth/BTCUSDT/BTCUSDT-depth-YYYY-MM-DD.zip`